In [1]:
import datetime
import numpy as np
import pandas as pd
from scipy.special import expit

In [2]:
# Coordinates for Soroti, Uganda
latitude = 1.7157
longitude = 33.6117

start_date = '2021-01-01'
end_date = '2022-01-01'

days = pd.date_range(start=start_date, end=end_date, freq='h')[:-1]
days

DatetimeIndex(['2021-01-01 00:00:00', '2021-01-01 01:00:00',
               '2021-01-01 02:00:00', '2021-01-01 03:00:00',
               '2021-01-01 04:00:00', '2021-01-01 05:00:00',
               '2021-01-01 06:00:00', '2021-01-01 07:00:00',
               '2021-01-01 08:00:00', '2021-01-01 09:00:00',
               ...
               '2021-12-31 14:00:00', '2021-12-31 15:00:00',
               '2021-12-31 16:00:00', '2021-12-31 17:00:00',
               '2021-12-31 18:00:00', '2021-12-31 19:00:00',
               '2021-12-31 20:00:00', '2021-12-31 21:00:00',
               '2021-12-31 22:00:00', '2021-12-31 23:00:00'],
              dtype='datetime64[ns]', length=8760, freq='h')

## Seting up models

In [3]:
#@title Day of the Year
def day_of_the_year(date):
    """
    Returns the day of the year

    Parameters
    ----------
    date : datetime object
        date of interest

    Returns
    -------
    day : int
        day of the year (1 to 365)
    """
    if isinstance(date, datetime.datetime):
        return date.timetuple().tm_yday
    else:
        msg = "date must be a datetime object or array of datetime objects"
        raise TypeError(msg)


In [ ]:
Days_of_the_year = [day_of_the_year(day) for day in days]
Days_of_the_year[350:]

[15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 15,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 16,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 17,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 18,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 19,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 20,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 21,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 22,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,
 23,


### Angle of Declination

This is the angular position of the sun at the solar noon (local meridian) with respect to the plane of equator and usually vary between -23.45 and 23.45. Using Cooper equation:

\begin{align}
\delta = 23.45\sin(360\frac{284 + DoY}{365})
\end{align}
where DoY is the day of the year

In [5]:
def declination(day : datetime):
  """
  this function finds solar declination in degrees using Cooper fomrular

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  delta : float
      solar declination in degrees
  """
  day_of_year = day_of_the_year(day)
  delta = 23.45 * np.sin(np.deg2rad(360 * (284 + day_of_year) / 365))
  return delta

### Hour angel (ω)
This is the angular displacement of the sun east or west of local medrian due to rotation of the earth arount its axis.

\begin{align}
ω = 15(solar time - 12)
\end{align}
Where
\begin{align}
solar time = standard time + E + 4(lon)
\end{align}
and
\begin{align}
E = 9.87\cos(2B) - 7.53\sin(B) - 1.5\sin(B)
\end{align}
where B is day angle gven by:
\begin{align}
B = (DoY - 81)\frac{360}{365}
\end{align}

In [6]:
def day_angle(day : datetime):
  """
  this function finds day angle in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  B : float
      day angle in degrees
  """
  day_of_year = day_of_the_year(day)
  B = (day_of_year - 81) * (360 / 365)
  return B

In [7]:
def solar_time(day : datetime):
  """
  this function finds solar time in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  E : float
      solar time in degrees
  """
  day_of_year = day_of_the_year(day)
  B = day_angle(day)
  E = 9.87 * np.cos(np.deg2rad(2 * B)) - 7.53 * np.sin(np.deg2rad(B)) - 1.5 * np.sin(np.deg2rad(B))
  return day.hour + day.minute / 60 + (E + 4*longitude) / 60

In [8]:
def hour_angle(day : datetime):
  """
  this function finds hour angle in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  ω : float
      hour angle in degrees
  """
  return 15 * (solar_time(day) - 12)

### Angle of Elevation (h)

This is the angle between the sun ray tot he obeserver and the projection on the horizontal plane

\begin{align}
  \sin(h) = \sin(\delta)\sin(Φ) + \cos(δ)\cos(Φ)\cos(ω)
\end{align}

where
* h is angle of elevation
* δ is angle of declination
* Φ is latitude
* ω is hour angle

In [9]:
def angle_of_elevation(day : datetime, latitude : float):
  """
  this function finds angle of elevation in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  h : float
      angle of elevation in degrees
  """
  delta = declination(day)
  hour_ang = hour_angle(day)

  h = np.arcsin(np.sin(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) *
                np.cos(np.deg2rad(hour_ang)))

  return np.rad2deg(h)

### Solar azimuth (γ)
This the angular displacement from south of the projection beam radiation on horizontal plane

\begin{align}
  \cos(γ) = \frac{\sin(h)\sin(Φ) - \sin(δ)}{\cos(h)\cos(Φ)}
\end{align}


In [10]:
def solar_azimuth(day : datetime, latitude : float):
  """
  this function finds solar azimuth in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest

  Returns
  -------
  azimuth : float
      solar azimuth in degrees
  """
  delta = declination(day)
  h = angle_of_elevation(day, latitude)

  azimuth = np.arccos((np.sin(np.deg2rad(h)) * np.sin(np.deg2rad(latitude)) -
                np.sin(np.deg2rad(delta))) / (np.cos(np.deg2rad(h)) *
                np.cos(np.deg2rad(latitude))))

  return np.rad2deg(azimuth)

### Angle of incidence

This is the angle between the sun ray on the surface and the normal to the plane of incidennce

\begin{align}
  \sin(ν) = \sin(δ)\sin(Φ)\cos(α) - \sin(δ)\cos(Φ)\sin(α)\cos(β) + \cos(δ)\cos(Φ)\cos(α)\cos(ω) + \cos(δ)\sin(Φ)\sin(α)\cos(β)\cos(ω) + \cos(δ)\sin(α)\sin(β)\sin(ω)
\end{align}

where:
* ν is angle of incidence
* α is surface tilt
* β is surface azimuth

In [11]:
def angle_of_incidence(day : datetime, latitude : float, surface_azimuth : float, surface_tilt : float):
  """
  this function finds angle of incidence in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest
  surface_azimuth : float
      surface azimuth in degrees
  surface_tilt : float
      surface tilt in degrees

  Returns
  -------
  aoi : float
      angle of incidence in degrees
  """
  delta = declination(day)
  hour_angle = hour_angle(day)

  aoi = np.arcsin(np.sin(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) * np.cos(np.deg2rad(surface_tilt)) -
                np.sin(np.deg2rad(delta)) * np.cos(np.deg2rad(latitude)) * np.sin(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(surface_azimuth)) +
                np.cos(np.deg2rad(delta)) * np.cos(np.deg2rad(latitude)) * np.cos(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(hour_angle)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(latitude)) * np.sin(np.deg2rad(surface_tilt)) * np.cos(np.deg2rad(surface_azimuth)) * np.cos(np.deg2rad(hour_angle)) +
                np.cos(np.deg2rad(delta)) * np.sin(np.deg2rad(surface_azimuth)) * np.sin(np.deg2rad(surface_tilt)) * np.sin(np.deg2rad(hour_angle)))

  return np.rad2deg(aoi)


## Extraterrestrial Radiation Models

### Spencer

\begin{align}
  G_{ET,h} = I_o(1.000110 + 0.034221\cos(B^*) + 0.001280\sin(B^*) + 0.000719\cos(2B^*) + 0.000077\sin(2B^*))\sin(h)
\end{align}

where
\begin{align}
  B^* = (DoY - 1)\frac{360}{365}
\end{align}



In [12]:
def B_star(day : datetime):
  """
  this function finds B* in degrees

  Parameters
  ----------
  day : datetime object
      date of interest

  Returns
  -------
  B_star : float
      B* in degrees
  """

  day_of_year = day_of_the_year(day)
  B_star = (day_of_year - 1) * (360 / 365)
  return B_star

In [13]:
def extra_irr(day : datetime, latitude : float):
  """
  this function finds extraterrestrial radiation in degrees

  Parameters
  ----------
  day : datetime object
      date of interest
  latitude : float
      latitude of interest

  Returns
  -------
  np array of G_ET_h values

  """

  solar_constant = 1367
  B_radians = np.deg2rad(B_star(day))
  h = angle_of_elevation(day, latitude)


  E_spencer = solar_constant * (1.000110 + 0.034221 * np.cos(B_radians) + 0.00128 * np.sin(B_radians) + 0.000719 * np.cos(2 * B_radians) + 0.000077 * np.sin(2 * B_radians)) * np.sin(h)

  return E_spencer

## Decomposition Model

Decomposition model estimates the fraction beam and diffuse horizontal irradiance from measured global horizontal irradiance. The cardinal input of these models are GHI and clearnes index kt. Clearness index is the function which compute global horizontal clearness index with respect to the extraterestrial irradiance on horizontal plane

\begin{align}
  K_t = \frac{G_h}{G_{ET,h}}
\end{align}

Where Kd is diffuse fraction

\begin{align}
  K_d = \frac{G_{d, h}}{G_h}
\end{align}


### Boland

\begin{align}
  K_d = \frac{1}{1 + e^{7.997(K_t - 0.587)}}
\end{align}

`np.exp()` RuntimeWarning: overflow encountered 

Implementing the equation as a signmoid function using `scipy.special.expit`
\begin{align}
  expit(x) = \frac{1}{1 + e^{-x}}
\end{align}

In [40]:
def boland(times : datetime, latitude : float, GHI : float):
    """
    this function finds extraterrestrial radiation in degrees

    Parameters
    ----------
    time : datetime object
        date of interest
        latitude : float
        latitude of interest
    GHI : float
        global horizontal irradiance
    Returns
    -------
    np array of G_ET_h values
    """

    Extr = np.array([extra_irr(time, latitude) for time in times])
    kt = GHI / Extr
    kt = np.where(kt < 0, 0.587, kt)
    k_d = expit(-7.997 * (kt - 0.587))
    k_d = k_d.reshape(-1, 1)

    GHI = GHI.to_numpy()[:, np.newaxis]
    BHI = GHI * (1 - k_d)

    return BHI

## Implementation of Models

In [15]:
df = pd.DataFrame({'Day': days})
df['Day'] = df['Day'].dt.tz_localize('Africa/Kampala') # Set timezone to Kampala
df.head()

,Day
0,2021-01-01 00:00:00+03:00
1,2021-01-01 01:00:00+03:00
2,2021-01-01 02:00:00+03:00
3,2021-01-01 03:00:00+03:00
4,2021-01-01 04:00:00+03:00


In [16]:
df['GMT_time'] = [pd.to_datetime(x).tz_convert('GMT') for x in df['Day']]  
df.head()

,Day,GMT_time
0,2021-01-01 00:00:00+03:00,2020-12-31 21:00:00+00:00
1,2021-01-01 01:00:00+03:00,2020-12-31 22:00:00+00:00
2,2021-01-01 02:00:00+03:00,2020-12-31 23:00:00+00:00
3,2021-01-01 03:00:00+03:00,2021-01-01 00:00:00+00:00
4,2021-01-01 04:00:00+03:00,2021-01-01 01:00:00+00:00


In [17]:
# calculate clearsky GHI
df['GHI_clearsky'] = df['GMT_time'].apply(extra_irr, latitude=latitude)
df.head(20)

,Day,GMT_time,GHI_clearsky
0,2021-01-01 00:00:00+03:00,2020-12-31 21:00:00+00:00,-1128.246614
1,2021-01-01 01:00:00+03:00,2020-12-31 22:00:00+00:00,-1103.483209
2,2021-01-01 02:00:00+03:00,2020-12-31 23:00:00+00:00,-1169.936695
3,2021-01-01 03:00:00+03:00,2021-01-01 00:00:00+00:00,-1294.118830
4,2021-01-01 04:00:00+03:00,2021-01-01 01:00:00+00:00,-1399.978867
5,2021-01-01 05:00:00+03:00,2021-01-01 02:00:00+00:00,-1387.659523
6,2021-01-01 06:00:00+03:00,2021-01-01 03:00:00+00:00,-1178.455319
7,2021-01-01 07:00:00+03:00,2021-01-01 04:00:00+00:00,-767.165417
8,2021-01-01 08:00:00+03:00,2021-01-01 05:00:00+00:00,-237.270954
9,2021-01-01 09:00:00+03:00,2021-01-01 06:00:00+00:00,283.662240


In [18]:
# Reomve negative values
def filter_negative(x):
    if x < 0:
        return 0
    else:
        return x

In [19]:
df['GHI_clearsky'] = df['GHI_clearsky'].apply(filter_negative)
df.head(20)

,Day,GMT_time,GHI_clearsky
0,2021-01-01 00:00:00+03:00,2020-12-31 21:00:00+00:00,0.000000
1,2021-01-01 01:00:00+03:00,2020-12-31 22:00:00+00:00,0.000000
2,2021-01-01 02:00:00+03:00,2020-12-31 23:00:00+00:00,0.000000
3,2021-01-01 03:00:00+03:00,2021-01-01 00:00:00+00:00,0.000000
4,2021-01-01 04:00:00+03:00,2021-01-01 01:00:00+00:00,0.000000
5,2021-01-01 05:00:00+03:00,2021-01-01 02:00:00+00:00,0.000000
6,2021-01-01 06:00:00+03:00,2021-01-01 03:00:00+00:00,0.000000
7,2021-01-01 07:00:00+03:00,2021-01-01 04:00:00+00:00,0.000000
8,2021-01-01 08:00:00+03:00,2021-01-01 05:00:00+00:00,0.000000
9,2021-01-01 09:00:00+03:00,2021-01-01 06:00:00+00:00,283.662240


In [46]:
df['BHI'] = boland(df['GMT_time'], latitude, df['GHI_clearsky'])
df['DHI'] = df['GHI_clearsky'] - df['BHI']

In [47]:
df.head(20)

,Day,GMT_time,GHI_clearsky,BHI,DHI
0,2021-01-01 00:00:00+03:00,2020-12-31 21:00:00+00:00,0.000000,0.000000,0.000000
1,2021-01-01 01:00:00+03:00,2020-12-31 22:00:00+00:00,0.000000,0.000000,0.000000
2,2021-01-01 02:00:00+03:00,2020-12-31 23:00:00+00:00,0.000000,0.000000,0.000000
3,2021-01-01 03:00:00+03:00,2021-01-01 00:00:00+00:00,0.000000,0.000000,0.000000
4,2021-01-01 04:00:00+03:00,2021-01-01 01:00:00+00:00,0.000000,0.000000,0.000000
5,2021-01-01 05:00:00+03:00,2021-01-01 02:00:00+00:00,0.000000,0.000000,0.000000
6,2021-01-01 06:00:00+03:00,2021-01-01 03:00:00+00:00,0.000000,0.000000,0.000000
7,2021-01-01 07:00:00+03:00,2021-01-01 04:00:00+00:00,0.000000,0.000000,0.000000
8,2021-01-01 08:00:00+03:00,2021-01-01 05:00:00+00:00,0.000000,0.000000,0.000000
9,2021-01-01 09:00:00+03:00,2021-01-01 06:00:00+00:00,283.662240,273.598871,10.063370
